In [ ]:
!pip install a2a-sdk httpx uvicorn

In [ ]:
import asyncio
import json
import uuid
import os
import logging

import httpx

from a2a.client import A2ACardResolver, A2AClient
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore e
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    MessageSendParams,
    SendMessageRequest,
    SendStreamingMessageRequest,
)
from a2a.utils import new_agent_text_message


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("A2A_Tutorial_Notebook")
logger.info("A2A SDK and libraries imported. Logging configured.")

NEWS_AGENT_BASE_URL = "http://localhost:9001"
EVENTS_AGENT_BASE_URL = "http://localhost:9002"

PUBLIC_AGENT_CARD_PATH = "/.well-known/agent.json"
EXTENDED_AGENT_CARD_PATH = "/agent/authenticatedExtendedCard"

logger.info(f"News Agent will be expected at: {NEWS_AGENT_BASE_URL}")
logger.info(f"Events Agent will be expected at: {EVENTS_AGENT_BASE_URL}")

In [ ]:
# news agent = agent logic class + agent executor + agent card & agent skills (agent server) + A2AStarletteApplication


from typing import Optional

# agent logic class
class NewsInfoAgent:

    async def get_latest_news(self, query: Optional[str]=None) -> str:
        logger.info(f"NewsInfoAgent received the query: {query}")
        return "Breaking News: AI discovers a new way to make coffee!"


# agent executor
class NewsInfoAgentExecutor(AgentExecutor):

    def __init__(self):
        super().__init__()
        self.agent = NewsInfoAgent()
        logger.info("NewsInfoAgentExecutor initialized.")

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue
    ) -> None:

        logger.info(f"NewsInfoAgentExecutor executing task: {context.task_id}")

        if context.request_message:
            logger.info(f"Request message content: {context.request_message.model_dump_json(indent=2)}")
        
        query_text = None
        if context.request_message and context.request_message.message and context.request_message.message.parts:
            for part in context.request_message.message.parts:
                if part.kind == 'text' and hasattr(part, 'text'):
                    query_text = part.text
                    logger.info(f"Extracted query from message: {query_text}")
                    break

        try:
            news_result = await self.agent.get_latest_news(query_text)
            event_queue.enqueue_event(new_agent_text_message(news_result))
            logger.info(f"NewsInfoAgentExecutor successfully sent news: {news_result}")
        
        except Exception as e:
            error_message = f"Error in NewsInfoAgentExecutor: {str(e)}"
            logger.error(error_message, exc_info=True)
            event_queue.enqueue_event(new_agent_text_message(f"Sorry, an error occured: {error_message}"))
        
        finally:
            event_queue.enqueue_event(None)

    
    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        logger.warning(f"NewsInfoAgentExecutorreceived cancel request for task: {context.task_id}")
        event_queue.enqueue_event(new_agent_text_message("Cancel operatio is not supported by this agent!"))
        event_queue.enqueue_event(None)


# agent skil
news_skill = AgentSkill(
    id="get_latest_news",
    name="Get Latest News",
    description="Provides the latest news headline.",
    tags=["news", "information", "tldr"],
    examples=["what is the news?", "latest headline", "give me news"]
)

# agent card
news_agent_card = AgentCard(
    name="News Information Agent",
    description="Provides news headlines for the TLDR of the day",
    url=NEWS_AGENT_BASE_URL,
    version="1.0.0",
    defaultInputModes=["text"],
    defaultOutputModes=["text"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[news_skill],
    supportsAuthenticatedExtendedCard=False
    )

logger.info(f"News Agent Card defined: {news_agent_card.name}")


news_agent_executor = NewsInfoAgentExecutor()
news_task_store = InMemoryTaskStore()
news_request_handler = DefaultRequestHandler(
    agent_executor=news_agent_executor,
    task_store=news_task_store
)


# create Starlette Application
news_agent_server_app = A2AStarletteApplication(
    agent_card=news_agent_card,
    http_handler=news_request_handler
).build()

logger.info(f"A2AStarletteApplication for News Agent created and built")

print("News Agent server configuration is ready. ",
      "See comments above on how to create a separate Python script to run it using uvicorn. ",
      f"It should listen on port 9001 as per NEWS_AGENT_BASE_URL ({NEWS_AGENT_BASE_URL}).")




if __name__ == '__main__':
     logger.info(f"Starting News Agent server on {NEWS_AGENT_BASE_URL}")
     uvicorn.run(news_agent_server_app, host='0.0.0.0', port=9001) # Port matches NEWS_AGENT_BASE_URL


In [ ]:
# events agent = agent logic class + agent executor + agent card & agent skills (agent server) + A2AStarletteApplication


from typing import Optional

# agent logic class
class EventsInfoAgent:

    async def get_current_events(self, query: Optional[str]=None) -> str:
        logger.info(f"EventsInfoAgen received the query: {query}")
        return "Current Event: The annual 'Innovate AI' conference is happening this week!"


# agent executor
class EventsInfoAgentExecutor(AgentExecutor):

    def __init__(self):
        super().__init__()
        self.agent = EventsInfoAgent()
        logger.info("EventsInfoAgentExecutor initialized.")

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue
    ) -> None:

        logger.info(f"EventsInfoAgentExecutor executing task: {context.task_id}")

        if context.request_message:
            logger.info(f"Request message content: {context.request_message.model_dump_json(indent=2)}")
        
        query_text = None
        if context.request_message and context.request_message.message and context.request_message.message.parts:
            for part in context.request_message.message.parts:
                if part.kind == 'text' and hasattr(part, 'text'):
                    query_text = part.text
                    logger.info(f"Extracted query from message: {query_text}")
                    break

        try:
            events_result = await self.agent.get_current_events(query_text)
            event_queue.enqueue_event(new_agent_text_message(events_result))
            logger.info(f"EventsInfoAgentExecutor successfully sent events: {events_result}")
        
        except Exception as e:
            error_message = f"Error in EventsInfoAgentExecutor: {str(e)}"
            logger.error(error_message, exc_info=True)
            event_queue.enqueue_event(new_agent_text_message(f"Sorry, an error occured: {error_message}"))
        
        finally:
            event_queue.enqueue_event(None)

    
    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        logger.warning(f"EventsInfoAgentExecutor received cancel request for task: {context.task_id}")
        event_queue.enqueue_event(new_agent_text_message("Cancel operation is not supported by this agent!"))
        event_queue.enqueue_event(None)


# agent skil
events_skill = AgentSkill(
    id="get_current_events",
    name="Get Current Events",
    description="Provides information about current events.",
    tags=['events', 'information', 'tldr', 'conference'],
    examples=['what are the current events?', 'any ongoing events?', 'tell me about events']
)

# agent card
events_agent_card = AgentCard(
    name="Current Events Information Agent",
    description="Provides updates on current events for the TLDR of the day.",
    url=EVENTS_AGENT_BASE_URL,
    version="1.0.0",
    defaultInputModes=["text"],
    defaultOutputModes=["text"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[events_skill],
    supportsAuthenticatedExtendedCard=False
    )

logger.info(f"Events Agent Card defined: {events_agent_card.name}")


events_agent_executor = EventsInfoAgentExecutor()
events_task_store = InMemoryTaskStore()
events_request_handler = DefaultRequestHandler(
    agent_executor=events_agent_executor,
    task_store=events_task_store
)


# create Starlette Application
events_agent_server_app = A2AStarletteApplication(
    agent_card=events_agent_card,
    http_handler=events_request_handler
).build()

logger.info(f"A2AStarletteApplication for Events Agent created and built")

print("Events Agent server configuration is ready. ",
      "See comments above on how to create a separate Python script to run it using uvicorn. ",
      f"It should listen on port 9002 as per EVENTS_AGENT_BASE_URL ({Events_AGENT_BASE_URL}).")




if __name__ == '__main__':
     logger.info(f"Starting Events Agent server on {Events_AGENT_BASE_URL}")
     uvicorn.run(events_agent_server_app, host='0.0.0.0', port=9002) # Port matches EVENTS_AGENT_BASE_URL


In [ ]:
# client side interaction with agents
    # discovering agents


import asyncio
import httpx

from a2a.client import A2ACardResolver

async def resolve_agent_cards:
    async with httpx.AsyncClient() as http_client:
        resolver = A2ACardResolver(http_client=http_client)

        print(f"Attempting to resolve News Agent card from {NEWS_AGENT_BASE_URL}")
        try:
            news_card = await resolver.get_public_card(NEWS_AGENT_BASE_URL)
            print("--- News Agent Card ---")
            print(news_card.model_dump_json(indent=2))
        except Exception as e:
            print(f"Could not resolve News Agent card: {e}")
            print("Please ensure the News Agent server is running on port 9001.")


        print(f"Attempting to resolve Events Agent card from {EVENTS_AGENT_BASE_URL}")
        try:
            events_card = await resolver.get_public_card(EVENTS_AGENT_BASE_URL)
            print("--- Events Agent Card ---")
            print(events_card.model_dump_json(indent=2))
        except Exception as e:
            print(f"Could not resolve Events Agent card: {e}")
            print("Please ensure the Events Agent server is running on port 9002.")


print("Define resolve_agent_cards() function. To execute, call 'await resolve_agent_cards()' in a new cell if your notebook supports it, or 'asyncio.run(resolve_agent_cards())' otherwise.")
print("IMPORTANT: Make sure your NewsInfoAgent (port 9001) and EventsInfoAgent (port 9002) servers are running before executing this.")

In [ ]:
# sending messages

import asyncio
import httpx
from uuid import uuid4

from a2a.client import A2AClient
from a2a.sdk.public_api import A2AClient, new_text_message_content_part


async def send_to_news_agent():
    async with httpx.AsyncClient() as http_client:
        a2a_client = A2AClient(http_client=http_client)

        task_id = f"client-task-{uuid4()}"
        message_parts = [new_text_message_content_part("What is the latest news headlines?")]

        print(f"Sending message to News Agent ({NEWS_AGENT_BASE_URL}) with task id: {task_id}")

        try:

            async for response_part in a2a_client.send(
                task_id=task,
                agent_url=NEWS_AGENT_BASE_URL,
                message_parts=message_parts
            ):
                if response_part:
                    print("---News Agent Response---")
                    print(response_part.model_dump_json(indent=2))
                else:
                    print("News Agent stream finished.")
        
        except Exception as e:
            print(f"Error sending message to News Agent: {e}")
            print("Please ensure the News Agent server is running on port 9001.")


print("Define send_to_news_agent() function. To execute, call 'await send_to_news_agent()' or 'asyncio.run(send_to_news_agent())'.")
print(f"IMPORTANT: Make sure your NewsInfoAgent server is running on {NEWS_AGENT_BASE_URL} before executing this.")


In [ ]:
# sending messages

import asyncio
import httpx
from uuid import uuid4

from a2a.client import A2AClient
from a2a.sdk.public_api import A2AClient, new_text_message_content_part


async def stream_from_events_agent():
    async with httpx.AsyncClient() as http_client:
        a2a_client = A2AClient(http_client=http_client)

        task_id = f"client-task-{uuid4()}"
        message_parts = [new_text_message_content_part("Any updates on ongoing events?")]

        print(f"Streaming from Events Agent ({EVENTS_AGENT_BASE_URL}) with task ID: {task_id}")

        try:
            message_count = 0
            async for response_part in a2a_client.send(
                task_id=task,
                agent_url=EVENTS_AGENT_BASE_URL,
                message_parts=message_parts
            ):
                if response_part:
                    message_count += 1
                    print("---Events Agent Response PArt {message_count}---")
                    print(response_part.model_dump_json(indent=2))
                else:
                    print("Events Agent stream finished.")
            if message_count == 0:
                print("No messages received from Events Agent before stream finished")
        
        except Exception as e:
            print(f"Error streaming from Events Agent: {e}")
            print("Please ensure the Events Agent server is running on port 9002.")


print("Define stream_from_events_agent() function. To execute, call 'await stream_from_events_agent()' or 'asyncio.run(stream_from_events_agent())'.")
print(f"IMPORTANT: Make sure your EventsInfoAgent server is running on {EVENTS_AGENT_BASE_URL} before executing this.")


In [ ]:
# create base agent class

from abc import ABC, abstractmethod
from typing import Dict, Any, Optional


class BaseInfoAgent(ABC):

    def __init__(self, agent_id:str, agent_type:str) -> None:
        
        self.agent_id = agent_id
        self.agent_type = agent_type

        print(f"{self.agent_type} with ID {self.agent_id} initialized.")

    @abstractmethod
    def _generate_tldr_part(self):
        pass

In [ ]:
# response task creation method

def create_a2a_text_part(text: str) -> Dict[str, Any]:
    return {"kind": "text", "text": text}

def create_a2a_artifact(name: str, parts: list): Dict[str, Any]:
    return {"name": name, "parts": parts}

def create_a2a_task(task_id: str, status: Dict[str, Any], artifacts: list) -> Dict[str, Any]:
    task = {
        "id": task_id,
        "status": status
    }
    if artifacts:
        task["artifacts"] = artifacts
    return task

def create_a2a_task_status(state: str, description: str) -> Dict[str, Any]:
    return {"state": state, "description": description}

def create_a2a_message(role: str, parts: list) -> Dict[str, Any]:
    return {"role": role, "parts": parts}

def generate_timestamp() -> str:
    from datetime import datetime
    return datetime.utcnow().isoformat() + "Z"


TASK_STATE_COMPLETED = "completed"
TASK_STATE_FAILED = "failed"
ROLE_AGENT = "agent"
ROLE_USER = "user"

def _create_response_task(self, task_id: str, tldr_content: str) -> Dict[str, Any]:
    tldr_text_part = create_a2a_text_part(tldr_content)
    tldr_artifact = create_a2a_artifact(name=f"{self.agent_id}_tldr_snippet.txt", parts=[tldr_text_part])

    response_task = create_a2a_task(
        task_id=task_id,
        status=create_a2a_task_status(
            TASK_STATE_COMPLETED,
            description="TLDR part generated successfully."
        ),
        artifacts=[tldr_artifact]
    )

    agent_response_message = create_a2a_message(
        role=ROLE_AGENT,
        parts=[create_a2a_text_part(
            f"{self.agent_id} has processed the request and generated its TLDR part."      
        )]
    )
    response_task["messages"] = [agent_response_message]
    response_task["lastUpdatedTimesStamp"] = generate_timestamp()

    return response_task


BaseInfoAgent._create_response_task = _create_response_task


In [ ]:
def handle_a2a_request(self, task_send_params: Dict[str, Any]) -> Dict[str, Any]:

    if not isinstance(task_send_params, dict):
        raise ValueError("task_send_params must be dictionary.")
    
    print(f"{self.agent_id} received task/send request:")
    print(json.dumps(task_send_params, indent=2))

    task_id = task_send_params.get("id")
    if not task_id:
        raise ValueError("Task ID is required in task_Send_params")

    try:
        tldr_content = self._generate_tldr_part()
        response_task = self._create_response_task(task_id, tldr_content)

        print(f"{self.agent_id} responding with Task object:")
        print(json.dumps(received, indent=2))
        return response_task
    
    except Exception as e:
        print(f"Error processing request in {self.agent_id}: {e}")
        error_task = create_a2a_task(
            task_id=task_id,
            status=create_a2a_task_status(
                TASK_STATE_FAILED,
                description=f"Failed to generate TLDR: {str(e)}"
            )
        )
        return error_task


BaseInfoAgent.handle_a2a_request = handle_a2a_request


In [ ]:
# create News Information Agent

class NewsInfoAgent(BaseInfoAgent):

    def __init__(self, agent_id: str) -> None:
        super().__init(agent_id, "NewsInfoAgent")

    def _generate_tldr_part(self) -> str:

        return (
            "Tech News: Google announces new Agent-to-Agent (A2A) communication protocol for AI systems. "
            "Sports: Liverpool FC clinches Premier League title after dramatic final match of the season."
        )

print("Testing NewsInfoAgent:")
test_news_agent = NewsInfoAgent("test-news-001")
print(f"Sample news content: {test_news_agent._generate_tldr_part()}")


In [ ]:
# create Events Information Agent

class EventsInfoAgent(BaseInfoAgent):

    def __init__(self, agent_id: str) -> None:
        super().__init(agent_id, "EventsInfoAgent")

    def _generate_tldr_part(self) -> str:

        return (
            "Local Events: City marathon scheduled for next Sunday, expect road closures. "
            "Arts & Culture: New exhibition opens at the Modern Art Museum featuring contemporary sculptors."
        )

print("Testing EventsInfoAgent:")
test_events_agent = EventsInfoAgent("test-events-001")
print(f"Sample events content: {test_events_agent._generate_tldr_part()}")


In [ ]:
# create the user-facing agent (coordinates with specialized agents to provide a complete TLDR)

class UserFacingAgent():
    
    def __init__(self, agent_id: str, news_agent: BaseInfoAgent, events_agent: BaseInfoAgent) -> None:

        self.agent_id = agent_id
        self.news_agent = news_agent
        self.events_agent = events_agent
        print(f"UserFacingAgent {self.agent_id} successfully __init__")


def _format_final_tldr(self, news_content, events_content) -> str:

    return ("Today's TLDR:\n"
    f"News:\n {news_content}\n"
    f"Events:\n {events_content}\n")

UserFacingAgent._format_final_tldr = _format_final_tldr

print("Testing TLDR formatting:")
sample_news = "Breaking: New AI protocol announced"
sample_events = "Concert tonight at 8 PM"
test_agent = UserFacingAgent("test-001", test_news_agent, test_events_agent)
formatted = test_agent._format_final_tldr(sample_news, sample_events)
print(formatted)

In [ ]:
# extract content from a2a task response

def _extract_content_from_task_response(self, task_response: Dict[str, Any], content_type: str) -> str:

    default_message = f"No {content_type} content received!"

    try:
        artifacts = task_response.get("artifacts", [])
        if not artifacts:
            return default_message

        first_artifact = artifacts[0]
        parts = first_artifact.get("parts", [])
        if no parts:
            return default_message
        
        return parts[0].get("text", default_message)
    
    except (TypeError, KeyError, IndexError) as e:
        print(f"Error extracting {content_type} content: {e}")
        return default_message


UserFacingAgent._extract_content_from_task_response = _extract_content_from_task_response


In [ ]:
# method for creating and sending a2a requests to other agents

def create_tasks_send_params(task_id: str, message_content: str) -> Dict[str, Any]:

    return {
        "id": task_id,
        "messages": [
            {
                "role": "user",
                "parts": [{"kind": "text", "text": message_content}]
            }
        ]
    }

def _request_agent_content(self, agent: BaseInfoAgent, content_type: str) -> Dict[str, Any]:

    task_id = f"{content_type}-subtask-{uuid.uuid4()}"
    task_params = create_tasks_send_params(
        task_id=task_id,
        message_content=f"Please provide the {content_type} TLDR part for today."
    )
    return agent.handle_a2a_request(task_params)

UserFacingAgent._request_agent_content = _request_agent_content



In [ ]:
# create comprehensive a2a task 

def _create_main_task_object(self, final_tldr: str, news_task_id: str, events_task_id: str) -> Dict[str, Any]:

    main_task_id = f"main-tldr-task-{uuid.uuid4()}"

    overall_task_artifact_part = create_a2a_text_part(final_tldr)
    overall_task_artifact = create_a2a_artifact("final_tldr_of_the_day.txt", [overall_task_artifact_part])

    main_task_status = create_a2a_task_status(
         TASK_STATE_COMPLETED,
        description="Successfully generated the daily TLDR."
    )

    initial_user_message = create_a2a_message(
        role=ROLE_USER,
        parts=[create_a2a_text_part("User to UserFacingAgent: Get me the TLDR of the day.")]
    )

    main_task = create_a2a_task(
        task_id=main_task_id,
        status=main_task_status,
        artifacts=[overall_task_artifact]
    )

    main_task["messages"] = [
        initial_user_message,
        create_a2a_message(
            role=ROLE_AGENT,
            parts=[create_a2a_text_part(f"UserFacingAgent used sub-task {news_task_id} for news.")]
        ),
        create_a2a_message(
            role=ROLE_AGENT,
            parts=[create_a2a_text_part(f"UserFacingAgent used sub-task {events_task_id} for events.")]
        )   
    ]
    main_task["lastUpdatedTimestamp"] = generate_timestamp()

    return main_task


UserFacingAgent._create_main_task_object = _create_main_task_object


In [ ]:
# orchestrate entire TLDR generation process

def get_daily_tldr(self) -> str:

    print(f"Agent {self.agent_id} starting to gather TLDR of the day...")

    news_task_response = self._request_agent_content(self.news_agent, "news")
    events_task_response = self._request_agent_content(self.events_agent, "events")

    news_content = self._extract_content_from_task_response(news_task_response, "news")
    events_content = self._extract_content_from_task_response(events_task_response, "events")

    final_tldr = self._format_final_tldr(news_content, events_content)

    print(f"Agent {self.agent_id} has compiled the final TLDR:")
    print(final_tldr)


    main_task = self._create_main_task_object(
        final_tldr,
        news_task_response.get("id", "unknown"),
        events_task_response.get("id", "unknown")
    )

    print(f"Overall main task object ({self.agent_id}):")
    print(json.dumps(main_task, indent=2))

    return final_tldr


UserFacingAgent.get_daily_tldr = get_daily_tldr

In [ ]:
# test

news_agent = NewsInfoAgent(agent_id="NewsBot-001")
events_agent = EventsInfoAgent(agent_id="EventOracle-XYZ")

user_facing_agent = UserFacingAgent(
    agent_id="ConciergeBot-7",
    news_agent=news_agent,
    events_agent=events_agent
)


try:

    daily_tldr_result = user_facing_agent.get_daily_tldr()
    print(daily_tldr_result)

except Exception as e:
    print(f"Demo failed with error: {e}")
    raise